In [ ]:
#| default_exp parallel

_ExecutionApprovalRequired: nbskill: refusing to execute unapproved cell id=cb524f32. Project policy /Users/macbook/Projects/nbskill/nbs/.nbskill-exec-approval.json did not approve this cell. Edit the policy entry, run it once yourself, or rerun nbskill with allow_new=True if you approve this source.

## Parallel notebook operations

Shared locks for MCP and notebook helpers. Calls touching the same notebook are serialized, calls touching different notebooks can proceed independently, and notebook execution uses one global semaphore.

Parallelism is useful only if notebook writes remain deterministic. Each notebook lock combines an in-process re-entrant lock with an OS file lock, held from read through commit and export. That makes concurrent MCP and Python writes to the same notebook wait for the active transaction to finish.

```python
with notebook_locks("nbs/01_read.ipynb", "nbs/02_write.ipynb"):
    ...

with execution_slot():
    ...
```

### Production contract

Concurrency is production infrastructure. Per-notebook locks must serialize operations on the same notebook, acquire multiple notebook locks in stable order, allow independent notebooks to proceed in parallel, and keep notebook execution behind one global execution slot.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import threading, time
from nbskill.parallel import execution_slot as _example_execution_slot
from nbskill.parallel import notebook_key as _example_notebook_key
from nbskill.parallel import notebook_locks as _example_notebook_locks
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook
import subprocess, sys, time

In [ ]:
#| eval: false
print(_example_notebook_key("nbs/../nbs/01_read.ipynb"))
with _example_notebook_locks("nbs/01_read.ipynb", "nbs/02_write.ipynb"):
    print("notebook locks acquired")
with _example_execution_slot():
    print("execution slot acquired")

In [ ]:
#| export
import fcntl, hashlib, threading, tempfile
from contextlib import contextmanager
from pathlib import Path

### Shared coordination state

The lock registry and execution gate live in one small module so every notebook operation uses the same coordination policy. Safe in-process execution is serialized here; wrappers that launch a child process keep notebook locks around the target paths.

In [ ]:
#| export
_LOCKS_GUARD = threading.Lock()
_NOTEBOOK_LOCKS = {}
_PROCESS_LOCKS_GUARD = threading.Lock()
_PROCESS_LOCKS = {}
_PROCESS_LOCK_ROOT = Path(tempfile.gettempdir()) / "nbskill-notebook-locks"
_EXECUTION_SEMAPHORE = threading.RLock()

### Stable path keys

Notebook paths can be relative, absolute, or not created yet. `notebook_key` normalizes them into a consistent lock key without requiring the file to already exist.

In [ ]:
#| export
def notebook_key(path):
    "Return a stable lock key for a notebook path."
    if path is None: return None
    return str(Path(path).expanduser().resolve(strict=False))

In [ ]:
#| export
def _notebook_lock(key):
    with _LOCKS_GUARD:
        lock = _NOTEBOOK_LOCKS.get(key)
        if lock is None:
            lock = threading.RLock()
            _NOTEBOOK_LOCKS[key] = lock
        return lock

def _process_lock_path(key):
    digest = hashlib.sha256(key.encode()).hexdigest()
    return _PROCESS_LOCK_ROOT / f"{digest}.lock"

def _acquire_process_lock(key):
    with _PROCESS_LOCKS_GUARD:
        current = _PROCESS_LOCKS.get(key)
        if current:
            _PROCESS_LOCKS[key] = (current[0], current[1] + 1)
            return
        _PROCESS_LOCK_ROOT.mkdir(parents=True, exist_ok=True)
        handle = _process_lock_path(key).open("a+")
        _PROCESS_LOCKS[key] = (handle, 1)
    try: fcntl.flock(handle, fcntl.LOCK_EX)
    except BaseException:
        with _PROCESS_LOCKS_GUARD: del _PROCESS_LOCKS[key]
        handle.close()
        raise

def _release_process_lock(key):
    with _PROCESS_LOCKS_GUARD:
        handle, depth = _PROCESS_LOCKS[key]
        if depth > 1:
            _PROCESS_LOCKS[key] = (handle, depth - 1)
            return
        del _PROCESS_LOCKS[key]
    fcntl.flock(handle, fcntl.LOCK_UN)
    handle.close()

### Per-notebook locks

`notebook_locks` acquires all requested notebook locks in sorted order. That lets operations touching multiple notebooks avoid deadlocks while still allowing unrelated notebooks to be edited in parallel.

In [ ]:
#| export
@contextmanager
def notebook_locks(*paths):
    "Acquire per-notebook locks in a stable order."
    keys = sorted({notebook_key(path) for path in paths if notebook_key(path) is not None})
    locks = [_notebook_lock(key) for key in keys]
    acquired, acquired_locks = [], []
    try:
        for lock in locks:
            lock.acquire()
            acquired_locks.append(lock)
        for key in keys:
            _acquire_process_lock(key)
            acquired.append(key)
        yield
    finally:
        for key in reversed(acquired): _release_process_lock(key)
        for lock in reversed(acquired_locks): lock.release()

### One execution slot

Notebook execution mutates kernels and outputs, so execution is serialized globally. Read and write operations can still use per-notebook locks around their own files.

In [ ]:
#| export
@contextmanager
def execution_slot():
    "Serialize notebook execution across parallel MCP calls."
    _EXECUTION_SEMAPHORE.acquire()
    try:
        yield
    finally:
        _EXECUTION_SEMAPHORE.release()

In [ ]:
root = demo_path("09_parallel_keys")
try:
    root.mkdir()
    a = root / "a.ipynb"
    b = root / "sub" / ".." / "b.ipynb"
    assert notebook_key(a).endswith("a.ipynb")
    assert notebook_key(b).endswith("b.ipynb")
    assert notebook_key(None) is None
finally:
    remove_demo_path(root)

In [ ]:
#| eval: false
with write_demo_notebook("09_parallel_same.ipynb") as path:
    entered = []
    def enter_same_lock():
        with notebook_locks(path):
            entered.append(True)
    with notebook_locks(path):
        thread = threading.Thread(target=enter_same_lock)
        thread.start()
        time.sleep(0.05)
        assert entered == []
    thread.join(timeout=1)
    assert entered == [True]

In [ ]:
#| hide
#| eval: false

def _locked_process_source(wait=False):
    body = "print('locked', flush=True)" + ("; sys.stdin.read()" if wait else "")
    return (
        "from nbskill.parallel import notebook_locks; import sys; "
        "lock = notebook_locks(sys.argv[1]); lock.__enter__(); "
        f"{body}; lock.__exit__(None, None, None)"
    )

path = demo_path("09_parallel_process.ipynb")
first = subprocess.Popen([sys.executable, "-c", _locked_process_source(True), str(path)], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
second = None
try:
    assert first.stdout.readline().strip() == "locked"
    second = subprocess.Popen([sys.executable, "-c", _locked_process_source(), str(path)], stdout=subprocess.PIPE, text=True)
    time.sleep(.1)
    assert second.poll() is None
    first.stdin.close()
    assert first.wait(1) == second.wait(1) == 0
    assert second.stdout.read().strip() == "locked"
finally:
    for process in first, second:
        if process and process.poll() is None: process.terminate()

In [ ]:
#| eval: false
active = 0
max_active = 0
guard = threading.Lock()

def hold_execution_slot():
    global active, max_active
    with execution_slot():
        with guard:
            active += 1
            max_active = max(max_active, active)
        time.sleep(0.05)
        with guard:
            active -= 1

threads = [threading.Thread(target=hold_execution_slot) for _ in range(3)]
for thread in threads: thread.start()
for thread in threads: thread.join()
assert max_active == 1

In [ ]:
assert _example_notebook_key("nbs/../nbs/01_read.ipynb") == _example_notebook_key("nbs/01_read.ipynb")
_acquired = False
with _example_notebook_locks("nbs/01_read.ipynb", "nbs/01_read.ipynb", None):
    _acquired = True
assert _acquired